# 📧 Reacher — Bulk Email Checker with SQLite + Google Drive

This notebook:
1. Installs the **check-if-email-exists** tool
2. Connects to your **SQLite database on Google Drive**
3. Reads all emails where `CHECK_BY_REACHER` is empty
4. Checks each email using **5 parallel threads** (one per domain group)
5. Saves the result back to your database

---

### ▶️ How to use
Click **Runtime → Run all** from the menu above.

---

### 📊 Result values saved to CHECK_BY_REACHER

| Value | Meaning |
|-------|---------|
| `safe` | Email is real and accepting messages |
| `risky` | Email exists but may bounce |
| `invalid` | Email does not exist |
| `unknown` | Could not verify — will not be re-checked |

In [ ]:
# ============================================================
#  STEP 1 — Install the Reacher Tool
# ============================================================

import urllib.request, json, subprocess, os

print('=' * 60)
print('  📦 STEP 1 of 4 — Installing Reacher tool...')
print('=' * 60)

api_url = 'https://api.github.com/repos/reacherhq/check-if-email-exists/releases/latest'
with urllib.request.urlopen(api_url) as r:
    data = json.loads(r.read())

version = data['tag_name']
print(f'  ✅ Latest version found: {version}')

download_url = None
filename = None
for asset in data['assets']:
    if 'linux' in asset['name'] and 'musl' in asset['name']:
        download_url = asset['browser_download_url']
        filename = asset['name']
        break

if not download_url:
    print('  ❌ Could not find Linux binary. Check the releases page manually.')
    raise SystemExit

print(f'  ✅ Found binary: {filename}')
urllib.request.urlretrieve(download_url, 'ciee.tar.gz')
print('  ✅ Download complete!')

result = subprocess.run(['tar', '-xzf', 'ciee.tar.gz'], capture_output=True, text=True)
if result.returncode == 0:
    print('  ✅ Extraction complete!')
else:
    print('  ❌ Extraction failed:', result.stderr)
    raise SystemExit

os.chmod('check_if_email_exists', 0o755)
print('  ✅ Tool installed and ready!')
print()

In [ ]:
# ============================================================
#  STEP 2 — Run Auto-Test to Confirm Tool is Working
# ============================================================

import subprocess, json

TEST_EMAIL = 'test@gmail.com'

print('=' * 60)
print('  🧪 STEP 2 of 4 — Running auto-test...')
print('=' * 60)
print(f'  🔍 Testing with sample email: {TEST_EMAIL}')
print()

result = subprocess.run(['./check_if_email_exists', TEST_EMAIL], capture_output=True, text=True)

try:
    out = json.loads(result.stdout)
    reachable = out.get('is_reachable', 'unknown').upper()
    emoji = {'SAFE': '✅', 'RISKY': '⚠️', 'INVALID': '❌', 'UNKNOWN': '❓'}.get(reachable, '❓')
    print(f'  {emoji}  Overall Result : {reachable}')
    print(f'  📝  Valid Format  : {out.get("syntax", {}).get("is_valid_syntax", "N/A")}')
    print(f'  🌐  Domain Exists : {out.get("mx", {}).get("accepts_mail", "N/A")}')
    print(f'  📨  Deliverable   : {out.get("smtp", {}).get("is_deliverable", "N/A")}')
    print(f'  🚫  Disabled Acct : {out.get("smtp", {}).get("is_disabled", "N/A")}')
    print()
    print('  🎉 Tool is working correctly! Proceeding to database...')
except Exception:
    print('  ⚠️  Could not parse result. Raw output:')
    print(result.stdout or result.stderr)
    raise SystemExit

print()

In [ ]:
# ============================================================
#  STEP 3 — Connect to Google Drive & Load Database
# ============================================================

from google.colab import drive
import os

print('=' * 60)
print('  📂 STEP 3 of 4 — Connecting to Google Drive...')
print('=' * 60)

drive.mount('/content/drive')

# ------------------------------------------------------------
#  ✏️  SET YOUR DATABASE PATH HERE
#  Replace the path below with the actual location of your
#  SQLite file inside Google Drive.
#
#  Example:
#    If your file is in My Drive root:
#      /content/drive/My Drive/email_marketing.db
#
#    If your file is in a folder called "Data":
#      /content/drive/My Drive/Data/email_marketing.db
# ------------------------------------------------------------

DB_PATH = '/content/drive/My Drive/Personal - Database/EMAIL MARKETING - GARDENING, TREE PLANTING.db'   # <-- CHANGE THIS IF NEEDED

# ------------------------------------------------------------

if os.path.exists(DB_PATH):
    print(f'  ✅ Database found: {DB_PATH}')
else:
    print(f'  ❌ Database NOT found at: {DB_PATH}')
    print()
    print('  👉 Please update the DB_PATH variable above with the')
    print('     correct path to your SQLite file in Google Drive.')
    raise SystemExit

print()

In [ ]:
# ============================================================
#  STEP 4 — Check Emails from Database Using 5 Threads
# ============================================================

import sqlite3, subprocess, json, threading, time, random
from datetime import datetime

# ------------------------------------------------------------
#  ⏱️  DELAY SETTINGS — Easily adjust the wait time between
#  each email check (in seconds) per thread.
# ------------------------------------------------------------

MIN_DELAY_SECONDS = 2    # <-- Minimum wait between checks
MAX_DELAY_SECONDS = 5    # <-- Maximum wait between checks

# ------------------------------------------------------------

TABLE_NAME  = 'email_marketing'
COL_EMAIL   = 'EMAIL'
COL_RESULT  = 'CHECK_BY_REACHER'

# Domain groups for each thread
DOMAIN_GROUPS = {
    'Thread-1 (Gmail)'           : ['gmail.com'],
    'Thread-2 (Hotmail/Outlook)' : ['hotmail.com', 'outlook.com', 'live.com', 'msn.com'],
    'Thread-3 (Yahoo)'           : ['yahoo.com', 'ymail.com'],
    'Thread-4 (AOL)'             : ['aol.com'],
    'Thread-5 (Other)'           : []   # catches everything not in threads 1-4
}

# All known domains (used to identify 'other')
KNOWN_DOMAINS = [
    d for group in list(DOMAIN_GROUPS.values())[:-1] for d in group
]

db_lock = threading.Lock()
print_lock = threading.Lock()
results_log = []   # stores (email, result) for final summary

# ---- Load all unchecked emails from database ----
print('=' * 60)
print('  📬 STEP 4 of 4 — Starting bulk email check...')
print('=' * 60)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute(f"""
    SELECT {COL_EMAIL} FROM {TABLE_NAME}
    WHERE {COL_RESULT} IS NULL OR TRIM({COL_RESULT}) = ''
""")
all_emails = [row[0] for row in cursor.fetchall()]
conn.close()

if not all_emails:
    print('  ℹ️  No unchecked emails found in the database.')
    print('      All emails already have a value in CHECK_BY_REACHER.')
    raise SystemExit

print(f'  📊 Total unchecked emails found: {len(all_emails)}')
print()

# ---- Assign emails to thread buckets ----
buckets = {name: [] for name in DOMAIN_GROUPS}

for email in all_emails:
    domain = email.split('@')[-1].lower() if '@' in email else ''
    assigned = False
    for thread_name, domains in list(DOMAIN_GROUPS.items())[:-1]:
        if domain in domains:
            buckets[thread_name].append(email)
            assigned = True
            break
    if not assigned:
        buckets['Thread-5 (Other)'].append(email)

for name, emails in buckets.items():
    print(f'  🧵 {name}: {len(emails)} emails')
print()

# ---- Function to check and save one email ----
def check_email(email, thread_name):
    try:
        result = subprocess.run(
            ['./check_if_email_exists', email],
            capture_output=True, text=True, timeout=30
        )
        out = json.loads(result.stdout)
        reachable = out.get('is_reachable', 'unknown').lower()
    except Exception:
        reachable = 'unknown'

    # Save to database with lock
    with db_lock:
        try:
            conn = sqlite3.connect(DB_PATH)
            conn.execute(f"""
                UPDATE {TABLE_NAME}
                SET {COL_RESULT} = ?
                WHERE {COL_EMAIL} = ?
            """, (reachable, email))
            conn.commit()
            conn.close()
            saved = True
        except Exception as e:
            saved = False

    emoji = {'safe': '✅', 'risky': '⚠️', 'invalid': '❌', 'unknown': '❓'}.get(reachable, '❓')

    with print_lock:
        print(f'  {emoji} [{thread_name}] {email} → {reachable.upper()}{"" if saved else " (save failed!"}')
        results_log.append((email, reachable))

# ---- Thread worker function ----
def thread_worker(thread_name, email_list):
    if not email_list:
        return
    for email in email_list:
        check_email(email, thread_name)
        delay = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
        time.sleep(delay)

# ---- Launch all 5 threads ----
print('  🚀 Launching all 5 threads...')
print('-' * 60)

threads = []
for thread_name, email_list in buckets.items():
    t = threading.Thread(
        target=thread_worker,
        args=(thread_name, email_list),
        name=thread_name
    )
    threads.append(t)
    t.start()

# Wait for all threads to finish
for t in threads:
    t.join()

print('-' * 60)
print()

# ============================================================
#  FINAL SUMMARY
# ============================================================

total   = len(results_log)
safe    = sum(1 for _, r in results_log if r == 'safe')
risky   = sum(1 for _, r in results_log if r == 'risky')
invalid = sum(1 for _, r in results_log if r == 'invalid')
unknown = sum(1 for _, r in results_log if r == 'unknown')

print('=' * 60)
print('  📊 SUMMARY — All Done!')
print('=' * 60)
print(f'  📬 Total emails checked : {total}')
print(f'  ✅ Safe                 : {safe}')
print(f'  ⚠️  Risky                : {risky}')
print(f'  ❌ Invalid              : {invalid}')
print(f'  ❓ Unknown              : {unknown}')
print()

# ---- Results Table ----
print('  📋 DETAILED RESULTS TABLE')
print('-' * 60)
print(f'  {"EMAIL":<40} {"RESULT"}')
print('-' * 60)
for email, result in sorted(results_log, key=lambda x: x[1]):
    emoji = {'safe': '✅', 'risky': '⚠️', 'invalid': '❌', 'unknown': '❓'}.get(result, '❓')
    print(f'  {email:<40} {emoji} {result.upper()}')
print('-' * 60)
print()
print('  💾 All results saved to your SQLite database.')
print('=' * 60)